# 02 - A/B test: gate at level 30 vs level 40 (module B)

**Question.** Does moving the first gate from level 30 to level 40 change retention?

**Plan.**
1. Check that the two groups have a similar size (sample ratio mismatch).
2. Test each retention indicator (day 1, day 7) with a two-proportion z-test.
3. Add confidence intervals (Wilson) and a bootstrap of the difference as a second opinion.

Two indicators are tested, so the significance threshold is 0.05 / 2 = **0.025** (Bonferroni correction).

In [1]:
import duckdb
import numpy as np
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

con = duckdb.connect("../game.duckdb", read_only=True)
df = con.sql("SELECT * FROM cookie_cats").df()   # 90,189 rows: small enough for pandas
con.close()

## 1. Sample ratio mismatch

With random assignment, both groups should have about the same size. A chi-square test compares the
observed sizes with an even split.

In [2]:
group_sizes = df["version"].value_counts().sort_index()
print(group_sizes)
print(chisquare(group_sizes))   # small p-value = the groups are not the same size

version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
Power_divergenceResult(statistic=np.float64(6.9024049496058275), pvalue=np.float64(0.008607987810836262))


## 2. Retention tests

For each indicator the cell prints: the z-test p-value, the retention of each version with its Wilson
95% interval, and the bootstrap interval of the difference *gate_40 - gate_30* (5,000 resamples, fixed seed).
The 97.5% interval is the one that matches the 0.025 threshold.

In [3]:
for metric in ["retention_1", "retention_7"]:
    # successes and group size per version
    g = df.groupby("version")[metric].agg(["sum", "count"])
    successes, n = g["sum"].values, g["count"].values

    _, p_value = proportions_ztest(successes, n)
    print(f"--- {metric} ---")
    print(f"p-value (z-test) = {p_value:.4f}")

    for version, s, total in zip(g.index, successes, n):
        low, high = proportion_confint(s, total, method="wilson")
        print(version, f"{s / total:.3%}", f"[{low:.3%} ; {high:.3%}]")

    # Bootstrap: resample each group with replacement and recompute the difference of rates
    rng = np.random.default_rng(42)
    a = df.loc[df.version == "gate_30", metric].astype(int).values
    b = df.loc[df.version == "gate_40", metric].astype(int).values
    diffs = [rng.choice(b, len(b)).mean() - rng.choice(a, len(a)).mean() for _ in range(5000)]
    print("Bootstrap gate_40 - gate_30, 95% interval  :", np.percentile(diffs, [2.5, 97.5]).round(5))
    print("Bootstrap gate_40 - gate_30, 97.5% interval:", np.percentile(diffs, [1.25, 98.75]).round(5))

--- retention_1 ---
p-value (z-test) = 0.0744
gate_30 44.819% [44.358% ; 45.280%]
gate_40 44.228% [43.772% ; 44.685%]


Bootstrap gate_40 - gate_30, 95% interval  : [-0.01222  0.00057]
Bootstrap gate_40 - gate_30, 97.5% interval: [-0.01304  0.0014 ]
--- retention_7 ---
p-value (z-test) = 0.0016
gate_30 19.020% [18.659% ; 19.387%]
gate_40 18.200% [17.848% ; 18.557%]


Bootstrap gate_40 - gate_30, 95% interval  : [-0.01328 -0.00323]
Bootstrap gate_40 - gate_30, 97.5% interval: [-0.01406 -0.00245]


## Conclusion

**Day 7.** Retention is 19.02% with gate_30 against 18.20% with gate_40 (about 0.8 points). The difference is
significant (p = 0.0016, threshold 0.025 for two indicators) and the bootstrap intervals exclude 0.

**Day 1.** The difference is not significant (p = 0.074): no conclusion. "Inconclusive" does not mean "no effect".

**Recommendation.** Keep the gate at level 30.

**Limits.** Retention is not revenue, and this dataset contains no revenue. The long-term effect is unknown.
The groups differ slightly in size (789 players, p = 0.009): a point of caution on the quality of the test.